In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup

def scrape_league_results(url, league_name):
    headers = {'User-Agent': 'Mozilla/5.0'}
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, 'html.parser')
    
    table = soup.find('table')
    df = pd.read_html(str(table))[0]
    
    df = df.dropna(subset=['Score'])
    df['Date'] = pd.to_datetime(df['Date'])
    df[['Home_Score', 'Away_Score']] = df['Score'].str.split('–', expand=True)
    
    columns = {
        'Home': 'Home_Team',
        'Away': 'Away_Team',
        'Home_Score': 'Home_Score',
        'Away_Score': 'Away_Score',
        'xG': 'Home_xG',
        'xG.1': 'Away_xG',
    }
    
    df = df[columns.keys()].rename(columns=columns)
    df['League'] = league_name
    
    df['Home_xG'] = pd.to_numeric(df['Home_xG'], errors='coerce')
    df['Away_xG'] = pd.to_numeric(df['Away_xG'], errors='coerce')
    
    return df

def scrape_all_leagues():
    leagues = {
        'Premier League': 'https://fbref.com/en/comps/9/schedule/Premier-League-Scores-and-Fixtures',
        'La Liga': 'https://fbref.com/en/comps/12/schedule/La-Liga-Scores-and-Fixtures',
        'Bundesliga': 'https://fbref.com/en/comps/20/schedule/Bundesliga-Scores-and-Fixtures',
        'Ligue 1': 'https://fbref.com/en/comps/13/schedule/Ligue-1-Scores-and-Fixtures',
        'Serie A': 'https://fbref.com/en/comps/11/schedule/Serie-A-Scores-and-Fixtures',
        'Serie B': 'https://fbref.com/en/comps/18/schedule/Serie-B-Scores-and-Fixtures',
        'Championship': 'https://fbref.com/en/comps/10/schedule/Championship-Scores-and-Fixtures'
    }
    
    dataframes = []
    for league_name, url in leagues.items():
        df = scrape_league_results(url, league_name)
        dataframes.append(df)
    
    combined_df = pd.concat(dataframes, ignore_index=True)
    combined_df.to_csv('Fixture_Results.csv', index=False)
    return combined_df

if __name__ == "__main__":
    results = scrape_all_leagues()
    print(f"Saved {len(results)} matches to CSV file")

Saved 2141 matches to CSV file
